# Credit Card Fraud Detection
## Phase 4: Autoencoder Experiments

In this phase, we run controlled experiments on the Autoencoder hyperparameters: latent dimension, architecture/dropout, learning rate, and batch size. We evaluate each experiment on the validation set using PR-AUC and F1-score to find the best configuration.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader, TensorDataset
import sys
import yaml
import time
import os

# Add src to path
sys.path.append('../')
from src.preprocessing import split_data_anomaly_detection, preprocess_features, load_config
from src.autoencoder import Autoencoder, get_device, train_autoencoder, compute_reconstruction_error
from src.evaluation import evaluate_anomaly_scores

os.makedirs('../reports/experiments', exist_ok=True)


### 1. Data Preparation

In [2]:
config = load_config('../configs/config.yaml')
df = pd.read_csv('../' + config['data']['subset_path'])

train_df, val_df, test_df = split_data_anomaly_detection(
    df, 
    val_size=config['data']['val_size'], 
    test_size=config['data']['test_size'], 
    random_state=config['random_seed']
)

(X_train, y_train), (X_val, y_val), (X_test, y_test), (scaler_time, scaler_amount) = preprocess_features(train_df, val_df, test_df)

device = get_device()
input_dim = X_train.shape[1]


Using CPU


### 2. Define Experiment Configurations
We will define a list of configurations to test. To keep this reproducible and fast, we'll test a few key variations.

In [3]:
experiments = [
    {'id': 'exp1_baseline', 'latent_dim': 14, 'dropout': 0.2, 'lr': 0.001, 'batch_size': 256, 'epochs': 30},
    {'id': 'exp2_small_latent', 'latent_dim': 7, 'dropout': 0.2, 'lr': 0.001, 'batch_size': 256, 'epochs': 30},
    {'id': 'exp3_high_dropout', 'latent_dim': 14, 'dropout': 0.5, 'lr': 0.001, 'batch_size': 256, 'epochs': 30},
    {'id': 'exp4_fast_lr', 'latent_dim': 14, 'dropout': 0.2, 'lr': 0.01, 'batch_size': 256, 'epochs': 30},
    {'id': 'exp5_large_batch', 'latent_dim': 14, 'dropout': 0.2, 'lr': 0.001, 'batch_size': 1024, 'epochs': 30},
]


### 3. Run Experiments
We will iterate over the configurations, train the model, and record the results on the validation set.

In [4]:
results = []

val_dataset = TensorDataset(torch.FloatTensor(X_val))

for exp in experiments:
    print(f"\n--- Running Experiment: {exp['id']} ---")
    # Update config dynamically for this run
    temp_config = config.copy()
    temp_config['training']['learning_rate'] = exp['lr']
    temp_config['training']['batch_size'] = exp['batch_size']
    temp_config['training']['epochs'] = exp['epochs']
    
    # DataLoaders
    train_dataset = TensorDataset(torch.FloatTensor(X_train))
    train_loader = DataLoader(train_dataset, batch_size=exp['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=exp['batch_size'], shuffle=False)
    
    # Init model
    model = Autoencoder(input_dim=input_dim, latent_dim=exp['latent_dim'], dropout=exp['dropout'])
    
    # Train
    start_time = time.time()
    model, train_losses, val_losses = train_autoencoder(model, train_loader, val_loader, temp_config, device)
    train_time = time.time() - start_time
    
    # Evaluate
    val_errors = compute_reconstruction_error(model, val_loader, device)
    metrics = evaluate_anomaly_scores(y_val, val_errors, plot=False)
    
    # Save results
    result = {
        'experiment_id': exp['id'],
        'latent_dim': exp['latent_dim'],
        'dropout': exp['dropout'],
        'lr': exp['lr'],
        'batch_size': exp['batch_size'],
        'epochs': len(train_losses), # Actual epochs run (early stopping)
        'train_time_sec': train_time,
        'val_loss_final': val_losses[-1],
        'pr_auc': metrics['pr_auc'],
        'best_f1': metrics['best_f1'],
        'best_precision': metrics['best_precision'],
        'best_recall': metrics['best_recall']
    }
    results.append(result)

results_df = pd.DataFrame(results)
display(results_df)



--- Running Experiment: exp1_baseline ---


Epoch 1/30 - Train Loss: 1.385156 - Val Loss: 2.075481


Epoch 2/30 - Train Loss: 1.147061 - Val Loss: 1.935632


Epoch 3/30 - Train Loss: 1.034064 - Val Loss: 1.827062


Epoch 4/30 - Train Loss: 0.983409 - Val Loss: 1.789805


...

Epoch 27/30 - Train Loss: 0.790692 - Val Loss: 1.617617


Epoch 28/30 - Train Loss: 0.786519 - Val Loss: 1.632121


Epoch 29/30 - Train Loss: 0.784253 - Val Loss: 1.604729


Epoch 30/30 - Train Loss: 0.790284 - Val Loss: 1.600539


,experiment_id,latent_dim,dropout,lr,batch_size,epochs,train_time_sec,val_loss_final,pr_auc,best_f1,best_precision,best_recall
0,exp1_baseline,14,0.2,0.001,256,17,28.046710,1.570395,0.771429,0.742754,0.669935,0.833333
1,exp2_small_latent,7,0.2,0.001,256,22,30.673738,1.598513,0.777728,0.742115,0.682594,0.813008
2,exp3_high_dropout,14,0.5,0.001,256,17,24.174160,2.069101,0.656276,0.633065,0.628000,0.638211
3,exp4_fast_lr,14,0.2,0.010,256,18,23.395400,1.560280,0.753353,0.725926,0.666667,0.796748
4,exp5_large_batch,14,0.2,0.001,1024,30,21.360707,1.600539,0.783099,0.749077,0.685811,0.825203


### 4. Save and Compare Results
We save the results to the reports directory for later analysis.

In [5]:
results_df.to_csv('../reports/experiments/autoencoder_results.csv', index=False)
print("Experiment results saved to reports/experiments/autoencoder_results.csv")

# Identify the best experiment based on PR-AUC
best_exp = results_df.loc[results_df['pr_auc'].idxmax()]
print(f"\nBest Experiment by PR-AUC: {best_exp['experiment_id']} (PR-AUC: {best_exp['pr_auc']:.4f})")


Experiment results saved to reports/experiments/autoencoder_results.csv

Best Experiment by PR-AUC: exp5_large_batch (PR-AUC: 0.7831)
